# 10 End to End - Rollout to Viewer Bundle

## Objective

Run the real FlyGym pipeline and package a deployable static viewer:

Simulation -> RolloutRecorder -> export_rollout() -> export_viewer_pose() -> viewer_bundle.zip.

The notebook does not start a browser or render Three.js inside Colab. It ends with a downloadable ZIP that can be unpacked and hosted as static files.

## Scientific boundary

The outputs are computational FlyGym observations and visualization artifacts. They are not evidence from real flies, biological validation, or a Parkinson disease diagnosis.

## Prerequisites

Use a fresh Python 3.12 Google Colab runtime with internet access. FlyGym 2.1.0 and MuJoCo 3.9.0 are installed from the project extra.

## Expected Output

The run creates `results/colab/Healthy_001/` and `results/colab/viewer_bundle.zip`.

## Troubleshooting

If a step fails, fix the reported environment or input problem and rerun from the beginning. Assertions stop the notebook before a partial bundle can be downloaded.

## Validation

The notebook checks finite normalized quaternions, strictly increasing timestamps, pose validation, bundle contents, and the static entrypoint.

## Next notebook

After downloading the bundle, open it through a static web server or deploy its unpacked contents to GitHub Pages.


## Step 1 - Install package


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/TanVi3001/drosophila-pd-flygym.git"
def is_repo_root(path):
    return (
        (path / "pyproject.toml").is_file()
        and (path / "scripts" / "build_viewer_bundle.py").is_file()
        and (path / "notebooks" / "colab").is_dir()
    )
start = Path.cwd().resolve()
repo = next((candidate for candidate in (start, *start.parents) if is_repo_root(candidate)), None)
if repo is None:
    target = start / "drosophila-pd-flygym"
    if not is_repo_root(target):
        subprocess.run(["git", "clone", REPO_URL, str(target)], check=True)
    repo = target.resolve()
os.chdir(repo)
assert (repo / "pyproject.toml").is_file(), f"pyproject.toml not found in {repo}"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[simulation]"], check=True)
src_path = str(repo / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import drosophila_pd
assert Path(drosophila_pd.__file__).exists()
print("Repository ready:", repo)
print("Package import OK:", drosophila_pd.__file__)


## Step 2 - Create Fly


In [ ]:
from drosophila_pd.flygym_adapter import FlyGymAdapter, FlyGymConfig

config = FlyGymConfig.from_yaml("configs/v2/flygym/healthy.yaml")
adapter = FlyGymAdapter()
fly = adapter.create_fly(config.fly)
assert fly is not None
assert getattr(fly, "name", None) == config.fly.name
print("Fly created:", fly.name)


## Step 3 - Create World


In [ ]:
world = adapter.create_world(config.world)
assert world is not None
print("World created:", type(world).__name__)


## Step 4 - Attach Fly and Create Simulation


In [ ]:
adapter.attach_fly(
    world,
    fly,
    position=config.world.spawn_position,
    orientation=config.world.spawn_orientation,
    add_ground_contact_sensors=config.world.add_ground_contact_sensors,
)
simulation = adapter.create_simulation(world, config.simulation)
assert simulation is not None
simulation.reset()
assert float(simulation.timestep) > 0
print("Fly attached to world")
print("Simulation created with timestep:", simulation.timestep)


## Step 5 - Record rollout


In [ ]:
from drosophila_pd.flygym_adapter import FlyGymRuntime, RolloutRecorder

step_count = 100
recorder = RolloutRecorder(
    simulation,
    fly.name,
    fly=fly,
    timestep=float(simulation.timestep),
    simulation_metadata={
        "dataset_id": "Healthy_001",
        "timestep_s": float(simulation.timestep),
        "source": "Colab end-to-end notebook",
    },
)
runtime = FlyGymRuntime(simulation, recorder=recorder, max_steps=step_count)
rollout = runtime.run()
assert rollout is recorder.rollout
assert runtime.current_step == step_count
assert rollout.frame_count == step_count + 1
print("Simulation steps:", runtime.current_step)
print("Recorded frames:", rollout.frame_count)


## Step 6 - Validate rollout


In [ ]:
import numpy as np

assert rollout.frame_count > 0
orientations = np.asarray([frame.orientation for frame in rollout.frames], dtype=float)
assert orientations.shape == (rollout.frame_count, 4)
orientation_norms = np.linalg.norm(orientations, axis=1)
assert np.isfinite(orientation_norms).all(), orientation_norms
assert np.all(orientation_norms > 0), orientation_norms
assert np.allclose(orientation_norms, 1.0, atol=1e-6), orientation_norms

timestamps = np.asarray([frame.timestamp_s for frame in rollout.frames], dtype=float)
assert np.isfinite(timestamps).all(), timestamps
assert np.all(np.diff(timestamps) > 0), timestamps
print("Quaternion norm min/max:", float(orientation_norms.min()), float(orientation_norms.max()))
print("Timestamp first/last:", float(timestamps[0]), float(timestamps[-1]))


## Step 7 - Export rollout


In [ ]:
from drosophila_pd.flygym_adapter import export_rollout

dataset_dir = repo / "results" / "colab" / "Healthy_001"
if dataset_dir.exists():
    shutil.rmtree(dataset_dir)
exported = export_rollout(rollout, dataset_dir)
assert dataset_dir.exists()
for key in ("rollout_json", "rollout_csv", "rollout_npz", "metadata", "manifest"):
    assert key in exported.files, key
    assert Path(exported.files[key]).exists(), exported.files[key]

arrays = np.load(exported.files["rollout_npz"])
assert "orientation" in arrays.files
assert "thorax_quaternions" in arrays.files
assert np.all(np.linalg.norm(arrays["orientation"], axis=1) > 0)
print("Rollout dataset exported:", dataset_dir)
print("NPZ arrays:", sorted(arrays.files))


## Step 8 - Export viewer pose


In [ ]:
from drosophila_pd.viewer_export import export_viewer_pose, validate_pose_document

viewer_pose_path = dataset_dir / "viewer_pose.json"
pose_result = export_viewer_pose(dataset_dir, viewer_pose_path)
assert viewer_pose_path.exists()
assert pose_result.validation.overall_pass, pose_result.validation.as_dict()

viewer_pose = json.loads(viewer_pose_path.read_text(encoding="utf-8"))
assert viewer_pose["frame_count"] == rollout.frame_count
assert viewer_pose["frame_count"] > 0
viewer_orientations = np.asarray([frame["orientation"] for frame in viewer_pose["frames"]], dtype=float)
viewer_norms = np.linalg.norm(viewer_orientations, axis=1)
assert np.all(viewer_norms > 0), viewer_norms
assert np.allclose(viewer_norms, 1.0, atol=1e-6), viewer_norms
viewer_times = np.asarray([frame["time"] for frame in viewer_pose["frames"]], dtype=float)
assert np.all(np.diff(viewer_times) > 0), viewer_times
assert validate_pose_document(viewer_pose).overall_pass
print("Viewer pose created:", viewer_pose_path)
print("Viewer frame count:", viewer_pose["frame_count"])


## Step 9 - Build viewer bundle


In [ ]:
def _bundle_repo_candidates(seed):
    seed = Path(seed).resolve()
    if seed.is_file():
        seed = seed.parent
    return (seed, *seed.parents, seed / "drosophila-pd-flygym")
known_seeds = [Path.cwd()]
for value in (globals().get("repo"), globals().get("viewer_pose_path")):
    if value is not None:
        known_seeds.append(Path(value))
bundle_repo = None
for seed in known_seeds:
    for candidate in _bundle_repo_candidates(seed):
        if (candidate / "scripts" / "build_viewer_bundle.py").is_file() and (candidate / "web").is_dir():
            bundle_repo = candidate
            break
    if bundle_repo is not None:
        break
assert bundle_repo is not None, "Could not locate repository containing scripts/build_viewer_bundle.py"
pose_candidates = []
for seed in known_seeds + [bundle_repo]:
    path = Path(seed).resolve()
    search_root = path if path.is_dir() else path.parent
    if search_root.is_dir():
        pose_candidates.extend(search_root.rglob("viewer_pose.json"))
pose_candidates = [path for path in set(pose_candidates) if path.is_file()]
assert pose_candidates, "No viewer_pose.json found for bundle build"
viewer_pose_path = max(pose_candidates, key=lambda path: (path.stat().st_mtime_ns, path.as_posix()))
repo = bundle_repo
os.chdir(repo)
bundle_zip = repo / "results" / "colab" / "viewer_bundle.zip"
build_command = [
    sys.executable,
    str(repo / "scripts" / "build_viewer_bundle.py"),
    "--pose",
    str(viewer_pose_path),
    "--web-root",
    str(repo / "web"),
    "--output",
    str(bundle_zip),
]
build_result = subprocess.run(
    build_command,
    cwd=str(repo),
    capture_output=True,
    text=True,
    check=False,
)
if build_result.stdout:
    print(build_result.stdout, end="")
if build_result.returncode != 0:
    if build_result.stderr:
        print(build_result.stderr, end="", file=sys.stderr)
    raise RuntimeError(
        "Viewer bundle build failed with exit code "
        f"{build_result.returncode}: {' '.join(build_command)}"
    )
bundle_dir = bundle_zip.parent / "viewer_bundle"
assert bundle_zip.exists()
assert bundle_dir.is_dir()
assert (bundle_dir / "index.html").exists()
assert (bundle_dir / "viewer_pose.json").exists()
assert (bundle_dir / "viewer" / "viewer.js").exists()
assert (bundle_dir / "web" / "index.html").exists()
print("Viewer bundle created:", bundle_zip)
print("Unpacked bundle:", bundle_dir)


## Step 10 - Download bundle


In [3]:
from zipfile import ZipFile

assert bundle_zip.is_file(), f"Step 9 did not create the bundle: {bundle_zip}"
with ZipFile(bundle_zip) as archive:
    names = set(archive.namelist())
assert "viewer_bundle/index.html" in names
assert "viewer_bundle/viewer_pose.json" in names
assert "viewer_bundle/viewer/viewer.js" in names
assert "viewer_bundle/manifest.json" in names
print("Bundle ready for download:", bundle_zip)
try:
    from google.colab import files
    files.download(str(bundle_zip))
except ImportError:
    print("Outside Colab. Download this file manually:", bundle_zip)


NameError: name 'bundle_zip' is not defined